# Universal Language — Python Walkthrough

This notebook demonstrates the **UL Forge** Python bindings: parse, validate, render, deparse, compose, analyze, and the new modal/force/pragmatic extensions.

**Prerequisites**: Build the Python module with `cd ul-forge && maturin develop -m crates/bindings/python/Cargo.toml`

## 1. Install & Import

In [ ]:
# If not already installed:
# !pip install maturin
# !cd ../ul-forge && maturin develop -m crates/bindings/python/Cargo.toml

import ul_forge
import json

# Helper to pretty-print JSON
def pp(obj):
    if isinstance(obj, str):
        obj = json.loads(obj)
    print(json.dumps(obj, indent=2))

## 2. Parse UL-Script → GIR

UL-Script is the textual notation. `parse()` converts it into a **GIR** (Geometric Intermediate Representation) — a typed graph with nodes and edges.

In [ ]:
# A simple predication: entity → relation → entity
script = "● → ●"
gir = ul_forge.parse(script)
pp(gir)

In [ ]:
# A more complex example: modified entity in an enclosure (concept)
script2 = "○{ @30 ● → ● }"
gir2 = ul_forge.parse(script2)
pp(gir2)

## 3. Validate

`validate()` runs a 4-layer validation pipeline:
1. **Schema** — well-formed JSON structure
2. **Sort** — type constraints (Entity/Relation/Modifier/Assertion)
3. **Invariant** — structural rules (e.g., no dangling edges)
4. **Geometry** — geometric coherence

In [ ]:
result = ul_forge.validate(json.dumps(gir))
pp(result)
print(f"\nValid: {result['valid']}")

## 4. Render → SVG

`render()` converts GIR to an SVG image. In Jupyter, we can display it inline.

In [ ]:
from IPython.display import SVG, display

svg = ul_forge.render(json.dumps(gir))
display(SVG(svg))

In [ ]:
# Render the enclosure example
svg2 = ul_forge.render(json.dumps(gir2))
display(SVG(svg2))

## 5. Round-Trip: Deparse

`deparse()` converts GIR back to canonical UL-Script. The output should be equivalent to the original input.

In [ ]:
canonical = ul_forge.deparse(json.dumps(gir))
print(f"Original:  {script}")
print(f"Canonical: {canonical}")

## 6. Analyze Structure

`analyze()` detects which Σ_UL operations are present in a GIR.

In [ ]:
analysis = ul_forge.analyze(json.dumps(gir2))
pp(analysis)
print(f"\nOperations detected: {analysis.get('operations', [])}")

## 7. Compose Operations

`compose()` applies one of the 13 Σ_UL operations to operands.

| Operation | Arity | Signature |
|-----------|-------|-----------|
| `negate` | 1 | a → a |
| `predicate` | 3 | e × r × e → a |
| `conjoin` | 2 | a × a → a |
| `embed` | 1 | a → e |
| ... | | 13 total |

In [ ]:
# Negate an assertion
assertion = ul_forge.parse("○{ ● → ● }")
negated = ul_forge.compose("negate", json.dumps([json.dumps(assertion)]))
pp(negated)

# Render the negation
display(SVG(ul_forge.render(json.dumps(negated))))

In [ ]:
# Conjoin two assertions
a1 = json.dumps(ul_forge.parse("○{ ● → ● }"))
a2 = json.dumps(ul_forge.parse("○{ ● ← ● }"))
conjoined = ul_forge.compose("conjoin", json.dumps([a1, a2]))
pp(conjoined)

## 8. Performative Force (Extension)

UL supports 6 illocutionary forces: `assert`, `query`, `direct`, `commit`, `express`, `declare`.

`set_force()` annotates a GIR with a performative force.

In [ ]:
# Parse a basic assertion
gir_assert = ul_forge.parse("○{ ● → ● }")

# Set it as a query
gir_query = ul_forge.set_force(json.dumps(gir_assert), "query")
pp(gir_query)
print(f"\nForce: {gir_query.get('force', 'not set')}")

## 9. Pragmatic Inference (Extension)

`infer_pragmatics()` runs inference rules on a surface GIR to derive intended meanings.

Rules include:
- **SI-1**: Scalar implicature ("some" → "not all")
- **QI-1**: Query intention
- **DI-1**: Directive intention

In [ ]:
# Infer pragmatics on the query
inferences = ul_forge.infer_pragmatics(json.dumps(gir_query))
pp(inferences)

print(f"\n{inferences['count']} inference(s) detected")
for inf in inferences.get('inferences', []):
    print(f"  [{inf['rule']}] {inf['surface']} → {inf['intended']}")

## 10. Full Pipeline

Putting it all together: parse → validate → analyze → set force → infer pragmatics → render.

In [ ]:
# Full pipeline
script = "○{ @45 ● → ● }"
print(f"Input: {script}\n")

# Parse
gir = ul_forge.parse(script)
print(f"1. Parsed: {len(gir.get('nodes', []))} nodes, {len(gir.get('edges', []))} edges")

# Validate
val = ul_forge.validate(json.dumps(gir))
print(f"2. Valid: {val['valid']}")

# Analyze
analysis = ul_forge.analyze(json.dumps(gir))
print(f"3. Operations: {analysis.get('operations', [])}")

# Set force
gir_forced = ul_forge.set_force(json.dumps(gir), "direct")
print(f"4. Force: direct")

# Infer pragmatics
pragmatics = ul_forge.infer_pragmatics(json.dumps(gir_forced))
print(f"5. Inferences: {pragmatics['count']}")

# Render
svg = ul_forge.render(json.dumps(gir))
print(f"6. SVG: {len(svg)} bytes")
display(SVG(svg))

---

## Next Steps

- Read the [UL Core writing system](../ul-core/writing-system/writing-system.md) to learn the notation
- Follow the [9-day curriculum](../docs/learning/curriculum.md) for structured learning
- Try the [practice exercises](../docs/learning/exercises.md)
- Explore the [formal foundations](../foundations/formal-foundations.md) for the mathematical specification